In [14]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
import pandas as pd
from ai_ledger_guard.rules import *

df = pd.read_csv("../data/raw/journal_entries_4203.csv", sep=";")

alerts = rule_weekend(df)
print(alerts.sum())

31537


In [16]:
import duckdb

In [17]:
duckdb.sql(
    """
    SELECT
        COUNT(Weekend)
    FROM df
    WHERE Weekend = 1 OR Weekend = 2
    """
)

┌────────────────┐
│ count(Weekend) │
│     int64      │
├────────────────┤
│          31537 │
└────────────────┘

In [18]:
is_saturday = df["weekend"] == 1
is_sunday = df["weekend"] == 2
print(df[is_saturday | is_sunday]["weekend"].value_counts().sum())
print((df["weekend"] > 0).sum())

31537
31537


In [19]:
differences = alerts != (df["weekend"] > 0)
print(differences.sum())


0


In [20]:
round_alert = rule_round_amount(df)
print(round_alert.sum())

466


In [21]:
rare_acc_alert = rule_rare_account(df)
print(rare_acc_alert.sum())

286


In [22]:
rare_accounts = duckdb.sql(
    """
    SELECT
        gl_account,
        gl_account_name,
        COUNT(*) AS lines
    FROM df
    GROUP BY gl_account, gl_account_name
    HAVING COUNT(*) < 100
    ORDER BY lines
    """
).df()

print(rare_accounts)
print(rare_accounts["lines"].sum())


    gl_account                    gl_account_name  lines
0         2020              Prior-period expenses      1
1          980                   Prepaid expenses      2
2         4360                 Insurance premiums      2
3         1700                  Other liabilities      2
4         4710                Packaging materials     24
5         4930                    Office supplies     24
6         4210            Rent immovable property     24
7         4250                           Cleaning     24
8         4240            Gas, electricity, water     24
9         4600               Advertising expenses     24
10        4950      Legal and consulting expenses     24
11        4580                Other vehicle costs     24
12        4260  Maintenance of operating premises     27
13        1200                               Bank     60
286


In [27]:
rare_accounts_with_anomalies = duckdb.sql(
    """
    SELECT
        gl_account,
        gl_account_name,
        COUNT(*) AS lines,
        SUM(label) AS anomalies
    FROM df
    GROUP BY gl_account, gl_account_name
    HAVING COUNT(*) < 100 AND SUM(label) > 0
    ORDER BY lines
    """
).df()

print(rare_accounts_with_anomalies)


   gl_account          gl_account_name  lines  anomalies
0        4210  Rent immovable property     24        2.0
1        4240  Gas, electricity, water     24        1.0
2        1200                     Bank     60       60.0
